# Second-Order System Response

**Learning Goals**
- Derive the standard form of a second-order system
- Define DC gain, natural undamped frequency, damping ratio, damped frequency, and exponential decay coefficient
- Compute and visualize the impulse, step, and ramp responses of a mass-spring-damper system
- Understand how the damping ratio governs the response type (underdamped, critically damped, overdamped)

## Relevant lecture video

In [1]:
from IPython.display import HTML

HTML('<iframe width="560" height="315" src="https://echo360.org/media/acf795af-ab19-49b0-ab44-e0bd53b6d2ec/public?autoplay=false&automute=false&currentMediaId=1af98332-4ecb-4a27-a417-3a71fda61a3b&sessionId=bbd41a1d-931c-4319-9af9-8757c869a1dc" frameborder="0" allowfullscreen></iframe>')

/home/matvei/JupyterBasedControlEngineeringTextbook/venv/lib/python3.12/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [1]:
%pip install -q ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import matplotlib.pyplot as plt

print("Libraries loaded.")

Libraries loaded.


---
## Standard Form of a Second-Order System

A second-order system has the standard transfer function:

$$H(s) = \frac{K \, \omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$$

### Parameters

| Symbol | Name | Description |
|--------|------|-------------|
| $K$ | **DC gain** | Steady-state output for a unit step input |
| $\omega_n$ | **Natural undamped frequency** (rad/s) | Oscillation frequency if there were no damping |
| $\zeta$ | **Damping ratio** | Determines how oscillations decay |

### Derived quantities

| Symbol | Name | Formula |
|--------|------|--------|
| $\omega_d$ | **Damped natural frequency** | $\omega_d = \omega_n \sqrt{1 - \zeta^2}$ |
| $\sigma$ | **Exponential decay coefficient** | $\sigma = \zeta \omega_n$ |

The poles of $H(s)$ are:

$$s_{1,2} = -\zeta\omega_n \pm j\omega_n\sqrt{1 - \zeta^2} = -\sigma \pm j\omega_d$$

### Damping regimes

| Condition | Type | Behavior |
|-----------|------|----------|
| $\zeta = 0$ | Undamped | Sustained oscillation at $\omega_n$ |
| $0 < \zeta < 1$ | Underdamped | Oscillatory decay at $\omega_d$ |
| $\zeta = 1$ | Critically damped | Fastest non-oscillatory return |
| $\zeta > 1$ | Overdamped | Slow, non-oscillatory decay |

In [3]:
K_2d = widgets.FloatSlider(min=0.1, max=5, step=0.1, value=1,
                           description='K (DC gain):',
                           style={'description_width': 'initial'})
wn_2d = widgets.FloatSlider(min=0.5, max=10, step=0.1, value=2,
                            description=r'$\\omega_n$ (rad/s):',
                            style={'description_width': 'initial'})
zeta_2d = widgets.FloatSlider(min=0.05, max=2, step=0.05, value=0.3,
                              description=r'$\\zeta$:',
                              style={'description_width': 'initial'})
input_2d = widgets.Dropdown(options=['impulse', 'step', 'ramp'],
                            value='step', description='Input:',
                            style={'description_width': 'initial'})
run_2d = widgets.Button(description='Plot', button_style='primary')
out_2d = widgets.Output()

def plot_2nd_order(K, wn, zeta, input_type):
    wd = wn * np.sqrt(abs(1 - zeta**2)) if zeta != 1 else 0
    t_max = max(10, 6 / (zeta * wn)) if zeta > 0 else 20
    t = np.linspace(0, t_max, 1000)

    fig, ax = plt.subplots(1, 1, figsize=(8, 4))

    if input_type == 'impulse':
        u = np.zeros_like(t)
        u[0] = 1 / (t[1] - t[0])
        if zeta == 0:
            y = K * wn * np.sin(wn * t)
        elif zeta == 1:
            y = K * wn**2 * t * np.exp(-wn * t)
        elif zeta > 1:
            sd = wn * np.sqrt(zeta**2 - 1)
            p1 = -zeta * wn + sd
            p2 = -zeta * wn - sd
            y = K * wn**2 / (2 * sd) * (np.exp(p1 * t) - np.exp(p2 * t))
        else:
            y = K * wn / np.sqrt(1 - zeta**2) * np.exp(-zeta * wn * t) * np.sin(wd * t)
        ax.set_title(f'Unit Impulse Response  (K={K}, $\\omega_n$={wn}, $\\zeta$={zeta})')
    elif input_type == 'step':
        u = np.ones_like(t)
        if zeta == 0:
            y = K * (1 - np.cos(wn * t))
        elif zeta == 1:
            y = K * (1 - (1 + wn * t) * np.exp(-wn * t))
        elif zeta > 1:
            sd = wn * np.sqrt(zeta**2 - 1)
            p1 = -zeta * wn + sd
            p2 = -zeta * wn - sd
            y = K * (1 - (p2 * np.exp(p1 * t) - p1 * np.exp(p2 * t)) / (p2 - p1))
        else:
            phi = np.arccos(zeta)
            y = K * (1 - np.exp(-zeta * wn * t) / np.sqrt(1 - zeta**2) * np.sin(wd * t + phi))
        ax.axhline(K, color='gray', linestyle='--', alpha=0.4, label='Steady state (K)')
        ax.set_title(f'Unit Step Response  (K={K}, $\\omega_n$={wn}, $\\zeta$={zeta})')
    else:  # ramp
        u = t
        if zeta == 0:
            y = K * (t - (1/wn) * np.sin(wn * t))
        elif zeta == 1:
            y = K * (t - 2/wn + (2/wn + t) * np.exp(-wn * t))
        elif zeta > 1:
            sd = wn * np.sqrt(zeta**2 - 1)
            p1 = -zeta * wn + sd
            p2 = -zeta * wn - sd
            y = K * (t - 2*zeta/wn + (wn**2/(2*sd)) * (np.exp(p1*t)/p1**2 - np.exp(p2*t)/p2**2))
        else:
            phi = np.arccos(zeta)
            y = K * (t - 2*zeta/wn + np.exp(-zeta*wn*t) / (wn*np.sqrt(1-zeta**2)) * np.sin(wd*t + 2*phi))
        ax.set_title(f'Unit Ramp Response  (K={K}, $\\omega_n$={wn}, $\\zeta$={zeta})')

    ax.plot(t, u, 'k--', linewidth=1, alpha=0.4, label='Input')
    ax.plot(t, y, 'b-', linewidth=2, label='Response')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Output')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

    # Annotate derived parameters
    info = f'$\\omega_d$ = {wd:.3f} rad/s,  $\\sigma$ = {zeta*wn:.3f}'
    ax.text(0.98, 0.05, info, transform=ax.transAxes,
            ha='right', va='bottom', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.show()

def on_run_2d(b):
    with out_2d:
        clear_output(wait=True)
        plot_2nd_order(K_2d.value, wn_2d.value, zeta_2d.value, input_2d.value)

run_2d.on_click(on_run_2d)
display(widgets.VBox([K_2d, wn_2d, zeta_2d, input_2d, run_2d, out_2d]))
print("Adjust parameters, select input type, then click Plot.")

Adjust parameters, select input type, then click Plot.


---
## Mass-Spring-Damper System

A mass $m$, damper $b$, and spring $k$ driven by force $f(t)$:

$$m\ddot{x}(t) + b\dot{x}(t) + kx(t) = f(t)$$

Taking the Laplace transform with zero initial conditions:

$$H(s) = \frac{X(s)}{F(s)} = \frac{1}{ms^2 + bs + k}$$

### Mapping to standard form

Comparing $H(s) = \dfrac{1/k}{\frac{m}{k}s^2 + \frac{b}{k}s + 1}$ with the standard form:

$$\omega_n = \sqrt{\frac{k}{m}}, \qquad
\zeta = \frac{b}{2\sqrt{km}}, \qquad
K = \frac{1}{k}$$

The derived parameters follow:

$$\omega_d = \omega_n\sqrt{1 - \zeta^2}, \qquad
\sigma = \zeta\omega_n = \frac{b}{2m}$$

In [4]:
m_msd = widgets.FloatSlider(min=0.5, max=10, step=0.1, value=2,
                            description='m (kg):',
                            style={'description_width': 'initial'})
b_msd = widgets.FloatSlider(min=0.1, max=10, step=0.1, value=1,
                            description='b (N\u00b7s/m):',
                            style={'description_width': 'initial'})
k_msd = widgets.FloatSlider(min=0.5, max=20, step=0.1, value=5,
                            description='k (N/m):',
                            style={'description_width': 'initial'})
input_msd = widgets.Dropdown(options=['impulse', 'step', 'ramp'],
                             value='step', description='Input:',
                             style={'description_width': 'initial'})
run_msd = widgets.Button(description='Plot', button_style='primary')
out_msd = widgets.Output()

def plot_msd(m, b, k, input_type):
    wn = np.sqrt(k / m)
    zeta = b / (2 * np.sqrt(k * m))
    K = 1 / k
    wd = wn * np.sqrt(abs(1 - zeta**2)) if zeta != 1 else 0

    t_max = max(10, 6 / (zeta * wn)) if zeta > 0 else 30
    t = np.linspace(0, t_max, 1000)

    fig, ax = plt.subplots(1, 1, figsize=(8, 4))

    if input_type == 'impulse':
        u = np.zeros_like(t)
        u[0] = 1 / (t[1] - t[0])
        if zeta == 0:
            y = K * wn * np.sin(wn * t)
        elif zeta == 1:
            y = K * wn**2 * t * np.exp(-wn * t)
        elif zeta > 1:
            sd = wn * np.sqrt(zeta**2 - 1)
            p1 = -zeta * wn + sd
            p2 = -zeta * wn - sd
            y = K * wn**2 / (2 * sd) * (np.exp(p1 * t) - np.exp(p2 * t))
        else:
            y = (K * wn / np.sqrt(1 - zeta**2)) * np.exp(-zeta * wn * t) * np.sin(wd * t)
        ax.set_title(f'MSD: Unit Impulse Response')
    elif input_type == 'step':
        u = np.ones_like(t)
        if zeta == 0:
            y = K * (1 - np.cos(wn * t))
        elif zeta == 1:
            y = K * (1 - (1 + wn * t) * np.exp(-wn * t))
        elif zeta > 1:
            sd = wn * np.sqrt(zeta**2 - 1)
            p1 = -zeta * wn + sd
            p2 = -zeta * wn - sd
            y = K * (1 - (p2 * np.exp(p1 * t) - p1 * np.exp(p2 * t)) / (p2 - p1))
        else:
            phi = np.arccos(zeta)
            y = K * (1 - np.exp(-zeta * wn * t) / np.sqrt(1 - zeta**2) * np.sin(wd * t + phi))
        ax.axhline(K, color='gray', linestyle='--', alpha=0.4, label=f'Steady state (K={K:.3f})')
        ax.set_title(f'MSD: Unit Step Response')
    else:  # ramp
        u = t
        if zeta == 0:
            y = K * (t - (1/wn) * np.sin(wn * t))
        elif zeta == 1:
            y = K * (t - 2/wn + (2/wn + t) * np.exp(-wn * t))
        elif zeta > 1:
            sd = wn * np.sqrt(zeta**2 - 1)
            p1 = -zeta * wn + sd
            p2 = -zeta * wn - sd
            y = K * (t - 2*zeta/wn + (wn**2/(2*sd)) * (np.exp(p1*t)/p1**2 - np.exp(p2*t)/p2**2))
        else:
            phi = np.arccos(zeta)
            y = K * (t - 2*zeta/wn + np.exp(-zeta*wn*t) / (wn*np.sqrt(1-zeta**2)) * np.sin(wd*t + 2*phi))
        ax.set_title(f'MSD: Unit Ramp Response')

    ax.plot(t, u, 'k--', linewidth=1, alpha=0.4, label='Input')
    ax.plot(t, y, 'b-', linewidth=2, label='Displacement x(t)')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('x (m)')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9)

    info = (f'$\\omega_n$={wn:.2f}, $\\zeta$={zeta:.3f}, $\\omega_d$={wd:.2f}, '
            f'$\\sigma$={zeta*wn:.3f}')
    ax.text(0.98, 0.05, info, transform=ax.transAxes,
            ha='right', va='bottom', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.show()

def on_run_msd(b):
    with out_msd:
        clear_output(wait=True)
        plot_msd(m_msd.value, b_msd.value, k_msd.value, input_msd.value)

run_msd.on_click(on_run_msd)
display(widgets.VBox([m_msd, b_msd, k_msd, input_msd, run_msd, out_msd]))
print("Adjust m, b, k, select input type, then click Plot.")

Adjust m, b, k, select input type, then click Plot.


---
## Summary

The **standard second-order system** is characterized by three parameters:
- **DC gain** $K$ scales the steady-state output
- **Natural undamped frequency** $\omega_n$ sets the oscillation speed (if any)
- **Damping ratio** $\zeta$ determines the response shape:
  - $\zeta = 0$: undamped, sustained oscillation
  - $0 < \zeta < 1$: underdamped, oscillatory decay at $\omega_d$
  - $\zeta = 1$: critically damped, fastest non-oscillatory
  - $\zeta > 1$: overdamped, sluggish response

From these we derive:
- **Damped frequency** $\omega_d = \omega_n\sqrt{1-\zeta^2}$ is the actual oscillation frequency
- **Exponential decay coefficient** $\sigma = \zeta\omega_n$ is how fast oscillations die out

The **mass-spring-damper** is the canonical example:
$$\omega_n = \sqrt{\frac{k}{m}}, \quad
\zeta = \frac{b}{2\sqrt{km}}, \quad
K = \frac{1}{k}$$